# Stage 8: residual-based fault / underperformance flagging

Results of `python -m src.faults`.

**Design.** A forecast that uses the station's own lagged power adapts to a lasting fault and hides it.
So the detector uses an **expected-power model without own-power inputs** (weather of the same hour, sun
position, calendar, station identity and metadata).

**Score per day and station:**
1. ratio = actual energy / expected energy over the bright hours (irradiance >= 200 W/m2);
2. divide by the station's normal ratio (median in the calibration period) and subtract 1;
3. 3-day rolling mean;
4. subtract the median of all stations on the same day (a weather-sensor error moves every station,
   a fault moves one).

A day is flagged when the score is below **-3 robust standard deviations** of the calibration scores.
Calibration = train + validation (all seasons), with out-of-fold expected power.

**Baseline detector:** the same recipe with a plain performance ratio (energy / irradiance).

**No fault labels exist**, so the detectors are scored on synthetic faults injected into the 2023 test
data, and the flags on unmodified data give an upper limit of the false-alarm rate.

In [ ]:
import pandas as pd
from IPython.display import Image

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
TABLES = "../results/tables/"
FIGURES = "../results/figures/"

## 1. The expected-power model
Error of the expected power (no own-power inputs). Train and validation are out-of-fold.

In [ ]:
pd.read_csv(TABLES + "08_expected_power_model.csv")

## 2. Thresholds and flags on unmodified test data
Most stations are almost never flagged; a few are flagged for a large part of 2023.

In [ ]:
pd.read_csv(TABLES + "08_false_alarms.csv").round(3).T

In [ ]:
Image(FIGURES + "08_score_spread.png")

## 3. Synthetic faults: recall, precision and delay
Precision assumes one fault per station-year and counts every flagged episode on unmodified data as a false alarm.

In [ ]:
det = pd.read_csv(TABLES + "08_fault_detection.csv")
det.pivot(index=["kind", "severity", "duration_days"], columns="detector", values="recall").round(3)

In [ ]:
det.pivot(index=["kind", "severity", "duration_days"], columns="detector", values="precision").round(3)

In [ ]:
det.pivot(index=["kind", "severity", "duration_days"], columns="detector", values="median_delay_days")

In [ ]:
Image(FIGURES + "08_recall_by_severity.png")

## 4. How strict should the threshold be?
A stricter threshold (more sigmas) flags fewer normal days but misses more mild faults.

In [ ]:
pd.read_csv(TABLES + "08_threshold_sensitivity.csv").round(3)

## 5. Why not use the forecast residual?
One injected fault seen by three residuals. The 1-h forecast uses lagged power, which drops with the fault, so its residual stays near 0.

In [ ]:
Image(FIGURES + "08_lag_vs_nowcast.png")

In [ ]:
pd.read_csv(TABLES + "08_lag_example.csv").round(3).T

## 6. Real flagged periods (candidates, not ground truth)
`has_known_zero_output_hours` = the period overlaps hours flagged in Stage 2 as zero output under strong sun.

In [ ]:
real = pd.read_csv(TABLES + "08_flagged_periods.csv")
len(real), real["has_known_zero_output_hours"].sum()

In [ ]:
real.head(15)

In [ ]:
real.groupby("station")["days"].agg(["count", "sum"]).sort_values("sum", ascending=False).head(13)